# Weather Underground PWS Data Pipeline

Fetch historical weather data from Weather Underground Personal Weather Stations (PWS) via API.

**Source:** https://api.weather.com/v2/pws (Weather Underground / The Weather Company PWS API)

You need **your own API key** (see `API_KEY_SETUP.md`) and must follow the Weather Underground /
The Weather Company terms of use, which restrict redistribution of the data. Without a key the
fetch cells are skipped and the notebook reads a CSV saved by an earlier run, if there is one.

## Key Features
- Historical hourly data (aggregated from minute-level observations)
- Clean column names with precipitation data prioritized
- Automatic chunking for date ranges > 31 days
- Quality checks and validation
- Multi-station support

## Pipeline Steps
1. Configure API key and stations
2. Fetch historical data (auto-chunked if needed)
3. Convert to clean DataFrames with standardized column names
4. Quality checks and data validation
5. Save results
6. Plot the data

## 1. Configuration

Configure API key, stations, date range, and output directory.

**Stations:**
- **Specific:** `STATION_IDS = ['KNYNEWYO1805', 'KNYNEWYO1850']` (list of PWS IDs).
- **All:** `STATION_IDS = 'all'` or `STATION_IDS = None` → load all from `dataset/meta/pws_metadata.csv`.

**Date Range:** Start and end dates (datetime objects).

**Output Directory:** `dataset/raw/fetched/wu/` (main dataset folder).

**API Key:** Set the environment variable `WU_API_KEY` (see `API_KEY_SETUP.md`). Do not write it into a tracked file.

In [1]:
# Setup: Import libraries and configure pipeline
from datetime import datetime, timedelta
from pathlib import Path
import sys
import warnings
warnings.filterwarnings('ignore')

# Add parent (fetch_data) to path for shared config
sys.path.insert(0, str(Path.cwd().parent))

# Shared config and WU functions (same pattern as asos_pipeline)
from config import OUTPUT_DIRS, PROJECT_ROOT
from wu_fetch import *
from wu_fetch import _disp  # project-relative paths for printing
import matplotlib.pyplot as plt
import pandas as pd

# ============================================================
# CONFIGURATION - EDIT THESE VALUES
# ============================================================
# Stations: list of IDs, or 'all' / None to load ALL from dataset/meta/pws_metadata.csv
STATION_IDS = ['KNYNEWYO1805', 'KNYNEWYO1850']  # specific stations
# STATION_IDS = 'all'   # or None — fetch ALL stations from pws_metadata.csv

START_DATE  = datetime(2024, 1, 1)
END_DATE    = datetime(2024, 1, 30)

# Output directory: main dataset folder (same as CLI/main.py)
OUTPUT_DIR = OUTPUT_DIRS['wu']
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
# ============================================================

# Resolve stations: specific list vs all from metadata
use_all = STATION_IDS is None or STATION_IDS == [] or STATION_IDS == 'all'
if use_all:
    pws_meta = read_pws_metadata()
    STATION_IDS = get_station_list(pws_meta)
    _station_source = "all (pws_metadata.csv)"
else:
    _station_source = "specific"


In [2]:

# Check API key status
api_status = check_api_key_status()
print(api_status['message'])

if api_status['available']:
    API_KEY = get_api_key()
else:
    API_KEY = None
    print("\nOptions to set API key:")
    print("  Environment variable: export WU_API_KEY='your_key'")
    print("See API_KEY_SETUP.md for detailed instructions.")
    print("Fetch cells will be skipped; a previously saved CSV is used if one exists.")

print(f'\nStations: {len(STATION_IDS)} ({_station_source})')
if _station_source == "specific":
    print(f'  {STATION_IDS}')
else:
    print(f'  First 5: {STATION_IDS[:5]}...')
print(f'Period: {START_DATE.date()} to {END_DATE.date()}')
print(f'Output directory: {_disp(OUTPUT_DIR)}')
print('\n✓ Setup complete')

⚠ WARNING: API key not found. Cannot fetch data. Set the WU_API_KEY environment variable (see API_KEY_SETUP.md)

Options to set API key:
  Environment variable: export WU_API_KEY='your_key'
See API_KEY_SETUP.md for detailed instructions.
Fetch cells will be skipped; a previously saved CSV is used if one exists.

Stations: 2 (specific)
  ['KNYNEWYO1805', 'KNYNEWYO1850']
Period: 2024-01-01 to 2024-01-30
Output directory: dataset/raw/fetched/wu

✓ Setup complete


## 2. Fetch Raw Data

Fetch historical weather data from Weather Underground API.

**Note:** Data is fetched in chunks if the date range exceeds 31 days (API limit). The pipeline automatically handles chunking and merging.

**Data Resolution:** Hourly (aggregated from minute-level PWS observations)

**API Key Required:** Weather Underground requires an API key for authentication.

### 2.0 API Test (Optional - Can Skip)

This cell tests the WU API to verify it's working correctly. You can skip this cell if you trust the API is functioning.

In [3]:
# Optional: Test API connection with a single station, single day
if API_KEY:
    print("Testing API connection...")
    test_result = get_historical_data_chunk(
        api_key=API_KEY,
        station_id=STATION_IDS[0],
        start_date=START_DATE,
        end_date=START_DATE + timedelta(days=1),
        units='m',
        data_type='hourly'
    )
    
    if test_result and 'observations' in test_result:
        print(f"✓ API test successful! Got {len(test_result['observations'])} observations")
        print(f"  Sample observation keys: {list(test_result['observations'][0].keys())[:5]}...")
    else:
        print("✗ API test failed or no data returned")
else:
    print("✗ API key not set - cannot test")

✗ API key not set - cannot test


### 2.1 Fetch Data

Fetch historical data for all configured stations.

In [4]:
# Fetch data using the pipeline
if not API_KEY:
    print("API key not found: skipping the fetch (see section 3 for the saved-data fallback).")
    results = None
else:
    results = run_wu_pipeline(
        api_key=API_KEY,
        station_ids=STATION_IDS,
        start_date=START_DATE,
        end_date=END_DATE,
        units='m',  # 'm' for metric, 'e' for imperial
        fetch_options={'historical': True},
        output_dir=None,  # We'll save manually later
        save_data=False
    )

API key not found: skipping the fetch (see section 3 for the saved-data fallback).


### 2.2 Quick Look at Raw Data

Inspect the raw data structure: columns, shape, and date ranges.

In [5]:
# Quick look at fetched data
if results and 'dataframes' in results:
    all_dfs = results['dataframes']
    
    for station_id, dfs in all_dfs.items():
        if 'clean' in dfs:
            df = dfs['clean']
            print(f"\n{station_id}:")
            print(f"  Shape: {df.shape}")
            print(f"  Columns: {list(df.columns)[:10]}...")
            
            # Find datetime column
            dt_col = None
            for col in ['datetime', 'time_local', 'obsTimeLocal']:
                if col in df.columns:
                    dt_col = col
                    break
            
            if dt_col:
                print(f"  Date range: {df[dt_col].min()} to {df[dt_col].max()}")
else:
    print("⚠ No data fetched")

⚠ No data fetched


## 3. Process Data

Collect the processed (clean) DataFrames: from the fetch above, or, when no API key is set, from
the most recent `WU_*.csv` saved by an earlier run in the output directory.

**Clean Data:** standardized column names; `precip_amount` is the hourly amount (mm), obtained by
differencing WU's running daily accumulation; `precip_rate` is in mm/h.

The data are raw station reports: no quality control is applied here.

In [6]:
# Extract processed data (clean DataFrames)
# precip_amount is the hourly amount (mm): run_wu_pipeline differences WU's running
# daily accumulation (adjust_precip_cumulative). precip_rate is a rate (mm/h) and is
# not summed.
processed_data = {}
data_source = None

if results and 'dataframes' in results:
    processed_data = {
        sid: dfs['clean']
        for sid, dfs in results['dataframes'].items()
        if 'clean' in dfs
    }
    data_source = 'api'
else:
    # No key / no fetch: read the most recent CSV saved by save_wu(), if any
    saved = sorted(OUTPUT_DIR.glob('WU_*.csv'), key=lambda p: p.stat().st_mtime)
    if saved:
        df_saved = pd.read_csv(saved[-1], parse_dates=['datetime'])
        processed_data = {sid: g.reset_index(drop=True)
                          for sid, g in df_saved.groupby('station_id')
                          if sid in STATION_IDS}
        data_source = f'saved file {saved[-1].name}'
    else:
        print(f"No API key and no saved WU_*.csv in {OUTPUT_DIR.relative_to(PROJECT_ROOT)}: nothing to process.")

if processed_data:
    print("=" * 60)
    print(f"PROCESSING RESULTS ({data_source})")
    print("=" * 60)
    print(f"\n✓ {len(processed_data)} station(s)")

    for station_id, df in processed_data.items():
        print(f"\n  {station_id}:")
        print(f"    Rows: {len(df):,}")
        if 'precip_amount' in df.columns:
            total_precip = df['precip_amount'].sum()
            rainy_hours = (df['precip_amount'] > 0).sum()
            print(f"    Total precip: {total_precip:.1f} mm ({rainy_hours} hours with precip > 0)")
        if 'temperature' in df.columns:
            print(f"    Temp range: {df['temperature'].min():.1f}°C to {df['temperature'].max():.1f}°C")


No API key and no saved WU_*.csv in dataset/raw/fetched/wu: nothing to process.


## 4. Save Data

Save processed data to CSV files (only data fetched in this run is saved).

**Filename Format:** `WU_YYYY-MM-DD_YYYY-MM-DD.csv`

In [7]:
# Save processed data
if processed_data and data_source == 'api':
    save_wu(
        processed_data=processed_data,
        output_dir=OUTPUT_DIR,
        overwrite=True
    )
else:
    print("Nothing new to save (no data fetched in this run)")

Nothing new to save (no data fetched in this run)


## 5. Visualization

Plot precipitation and weather data to visualize patterns and compare stations.

**Available plots:**
- Precipitation analysis (rate and cumulative)
- Multi-station comparison
- Weather dashboard (temperature, humidity, wind, pressure)

### 5.1 Precipitation Analysis

In [8]:
# Plot precipitation analysis for all stations
if processed_data:
    # Wrap DataFrames in expected format: {station_id: {'clean': df}}
    station_data_for_plot = {sid: {'clean': df} for sid, df in processed_data.items()}
    
    plot_precipitation_analysis_multi(
        station_data=station_data_for_plot,
        units='m',
        show=True
    )
else:
    print("⚠ No data to plot")

⚠ No data to plot


### 5.2 Cumulative Precipitation

In [9]:
# Plot cumulative precipitation
if processed_data:
    station_data_for_plot = {sid: {'clean': df} for sid, df in processed_data.items()}
    
    plot_cumulative_precipitation_multi(
        station_data=station_data_for_plot,
        units='m',
        show=True
    )
else:
    print("⚠ No data to plot")

⚠ No data to plot


### 5.3 Weather Dashboard (Single Station)

In [10]:
# Plot weather dashboard for first station
if processed_data:
    first_station = list(processed_data.keys())[0]
    df_first = processed_data[first_station]
    
    plot_weather_dashboard(
        df=df_first,
        station_id=first_station,
        units='m',
        show=True
    )
else:
    print("⚠ No data to plot")

⚠ No data to plot


## 6. Summary

Final summary of the pipeline execution.

In [11]:
# Summary
print("=" * 60)
print("SUMMARY")
print("=" * 60)
print(f"\nPeriod: {START_DATE.date()} to {END_DATE.date()}")
print(f"Stations: {len(processed_data)}")

if processed_data:
    # Show first few columns
    sample_cols = list(list(processed_data.values())[0].columns)[:8]
    print(f"\nVariables: {', '.join(sample_cols)}...")
    
    print(f"\nPrecipitation totals:")
    for station_id, df in processed_data.items():
        if 'precip_amount' in df.columns:
            total = df['precip_amount'].sum()   # hourly amounts, mm
            print(f"  {station_id}: {total:.1f} mm")
        else:
            print(f"  {station_id}: No precipitation data")

print(f"\nOutput directory: {_disp(OUTPUT_DIR)}")
print("\n" + "=" * 60)
print("✓ COMPLETE")

SUMMARY

Period: 2024-01-01 to 2024-01-30
Stations: 0

Output directory: dataset/raw/fetched/wu

✓ COMPLETE
